# Kanji Detector — Geração de Dataset + Treino YOLOv26n no Kaggle

**Checklist antes de rodar:**
1. Adicione o dataset **Manga109** (dataset cru original) no painel lateral direito do notebook em **+ Add Input**.
2. Habilite a GPU: *Session options → Accelerator → GPU T4 x2 ou P100*.
3. Clique em **Run All**.

In [ ]:
# Verificar GPU disponivel
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'GPU nao encontrada!')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponivel: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 0. Parâmetros e Configurações (Override)

Configure os parâmetros abaixo para sobrescrever as definições padrão do repositório. As variáveis de ambiente `KD_*` serão lidas pelo gerador e pelo treino.

In [ ]:
import os

# --- Parâmetros de Volume e Divisão ---
PAGES_AMOUNT    = 5000   # total de páginas sintéticas a gerar no Kaggle
VAL_RATIO       = 0.15   # fração das páginas reservada para validação
CROP_SIZE       = 640    # tamanho do crop em pixels

# --- Parâmetros do Modelo e Treino YOLO ---
YOLO_MODEL      = 'yolo26n.pt'
EPOCHS          = 50
BATCH           = 16
IMGSZ           = 640

# --- Layout de Texto Sintético ---
PROB_NEGATIVA   = 0.20   # fração de imagens sem caracteres (negativas)
N_BLOCOS_MIN    = 3
N_BLOCOS_MAX    = 8
N_COLUNAS_MIN   = 2
N_COLUNAS_MAX   = 6
CHARS_POR_COLUNA_MIN = 6
CHARS_POR_COLUNA_MAX = 16
BBOX_MARGEM     = 0.10   # expansão proporcional das bboxes (+10%)

# --- Probabilidades e Ranges de Distorções ---
BLUR_PROB       = 0.50
BLUR_SIGMA_MIN  = 0.1
BLUR_SIGMA_MAX  = 2.5

MORFO_PROB      = 0.40
MORFO_K_MIN     = 2
MORFO_K_MAX     = 4
MORFO_ITER_MIN  = 1
MORFO_ITER_MAX  = 3

RUIDO_PROB      = 0.60
RUIDO_STD_MIN   = 0.01
RUIDO_STD_MAX   = 0.25

ROTACAO_PROB    = 0.40
ROTACAO_MAX     = 3.0

# ============================================================
# Aplica as configurações enviando para o ambiente (os.environ)
# ============================================================
os.environ['KD_PAGES_AMOUNT']        = str(PAGES_AMOUNT)
os.environ['KD_VAL_RATIO']           = str(VAL_RATIO)
os.environ['KD_CROP_SIZE']           = str(CROP_SIZE)
os.environ['KD_YOLO_MODEL']          = str(YOLO_MODEL)
os.environ['KD_EPOCHS']              = str(EPOCHS)
os.environ['KD_BATCH']               = str(BATCH)
os.environ['KD_IMGSZ']               = str(IMGSZ)
os.environ['KD_PROB_NEGATIVA']       = str(PROB_NEGATIVA)
os.environ['KD_N_BLOCOS_MIN']        = str(N_BLOCOS_MIN)
os.environ['KD_N_BLOCOS_MAX']        = str(N_BLOCOS_MAX)
os.environ['KD_N_COLUNAS_MIN']       = str(N_COLUNAS_MIN)
os.environ['KD_N_COLUNAS_MAX']       = str(N_COLUNAS_MAX)
os.environ['KD_CHARS_POR_COLUNA_MIN'] = str(CHARS_POR_COLUNA_MIN)
os.environ['KD_CHARS_POR_COLUNA_MAX'] = str(CHARS_POR_COLUNA_MAX)
os.environ['KD_BBOX_MARGEM']         = str(BBOX_MARGEM)
os.environ['KD_BLUR_PROB']           = str(BLUR_PROB)
os.environ['KD_BLUR_SIGMA_MIN']      = str(BLUR_SIGMA_MIN)
os.environ['KD_BLUR_SIGMA_MAX']      = str(BLUR_SIGMA_MAX)
os.environ['KD_MORFO_PROB']          = str(MORFO_PROB)
os.environ['KD_MORFO_K_MIN']         = str(MORFO_K_MIN)
os.environ['KD_MORFO_K_MAX']         = str(MORFO_K_MAX)
os.environ['KD_MORFO_ITER_MIN']      = str(MORFO_ITER_MIN)
os.environ['KD_MORFO_ITER_MAX']      = str(MORFO_ITER_MAX)
os.environ['KD_RUIDO_PROB']          = str(RUIDO_PROB)
os.environ['KD_RUIDO_STD_MIN']       = str(RUIDO_STD_MIN)
os.environ['KD_RUIDO_STD_MAX']       = str(RUIDO_STD_MAX)
os.environ['KD_ROTACAO_PROB']        = str(ROTACAO_PROB)
os.environ['KD_ROTACAO_MAX']         = str(ROTACAO_MAX)

print('✅ Configurações de ambiente registradas!')

## 1. Instalar dependencias

In [ ]:
!pip install -q ultralytics albumentations fonttools Pillow opencv-python-headless pyyaml requests
print('Dependencias instaladas com sucesso.')

## 2. Configurar repositorio

In [ ]:
import os, sys

REPO_URL = 'https://github.com/MiguelMussalam/Detector-de-kanjis-n1.git'
WORK_DIR = '/kaggle/working'
REPO_DIR = os.path.join(WORK_DIR, 'Detector-de-kanjis-n1')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repositorio ja existe em {REPO_DIR}')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'Diretorio de trabalho: {os.getcwd()}')

## 3. Gerar Dataset Sintético no Kaggle

O script lerá as imagens originais do Manga109 sob `/kaggle/input/`, baixará as fontes TTF necessárias automaticamente e gerará 5.000 páginas contendo kanjis aplicados aleatoriamente com as caixas delimitadoras de treino YOLO.

In [ ]:
!python -m src.kanji_detector.generate_pages

## 4. Verificar arquivos gerados

In [ ]:
import os
from config import TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_LBL_DIR, VAL_LBL_DIR

train_imgs = os.listdir(TRAIN_IMG_DIR) if os.path.exists(TRAIN_IMG_DIR) else []
val_imgs   = os.listdir(VAL_IMG_DIR) if os.path.exists(VAL_IMG_DIR) else []
train_lbls = os.listdir(TRAIN_LBL_DIR) if os.path.exists(TRAIN_LBL_DIR) else []
val_lbls   = os.listdir(VAL_LBL_DIR) if os.path.exists(VAL_LBL_DIR) else []

print('Dataset sintético gerado em /kaggle/working/data/dataset:')
print(f'  Treino:    {len(train_imgs)} imagens | {len(train_lbls)} labels')
print(f'  Validacao: {len(val_imgs)} imagens | {len(val_lbls)} labels')

## 5. Treinar a YOLOv26n

In [ ]:
!python -m src.kanji_detector.train

## 6. Mostrar resultados e curvas de aprendizado

In [ ]:
from IPython.display import Image, display
import os

runs_dir = '/kaggle/working/Detector-de-kanjis-n1/kanji_detector/run1'
print(f'Procurando resultados em: {runs_dir}')

if os.path.exists(runs_dir):
    # Curvas de treino
    results_plot = os.path.join(runs_dir, 'results.png')
    if os.path.exists(results_plot):
        print('=== Curvas de Treino (mAP / Loss) ===')
        display(Image(results_plot))

    # Matriz de confusao
    conf_matrix = os.path.join(runs_dir, 'confusion_matrix.png')
    if os.path.exists(conf_matrix):
        print('=== Matriz de Confusao ===')
        display(Image(conf_matrix))

    # Amostras preditas
    val_batch = os.path.join(runs_dir, 'val_batch0_pred.jpg')
    if os.path.exists(val_batch):
        print('=== Predicoes no conjunto de validacao ===')
        display(Image(val_batch))
else:
    print('Diretorio de resultados nao encontrado.')

## 7. Exportar modelo treinado (`best.pt`)

In [ ]:
import shutil

runs_dir   = '/kaggle/working/Detector-de-kanjis-n1/kanji_detector/run1'
best_pt    = os.path.join(runs_dir, 'weights', 'best.pt')
output_dir = '/kaggle/working/output'

if os.path.exists(best_pt):
    os.makedirs(output_dir, exist_ok=True)
    shutil.copy2(best_pt, os.path.join(output_dir, 'best.pt'))
    print(f'Modelo best.pt exportado com sucesso para: {output_dir}/best.pt')
else:
    print('Arquivo best.pt nao encontrado. O treino terminou sem salvar?')

## 8. Compactar e Fazer Download de todos os Resultados

Este bloco cria um arquivo ZIP contendo os pesos (`best.pt`, `last.pt`), estatísticas de treino (`results.png`, `confusion_matrix.png`, `results.csv`, etc.) e as configurações usadas (`config.py`, `dataset.yaml`).

In [ ]:
import os
import zipfile
from IPython.display import FileLink, display

# Caminhos importantes
runs_dir = '/kaggle/working/Detector-de-kanjis-n1/kanji_detector/run1'
config_file = '/kaggle/working/Detector-de-kanjis-n1/config.py'
dataset_yaml = '/kaggle/working/dataset.yaml'  # yaml gerado para o kaggle
zip_name = '/kaggle/working/resultados_treino.zip'

def zipdir(path, ziph):
    for root, dirs, files in os.walk(path):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.join('run1_resultados', os.path.relpath(filepath, path))
            ziph.write(filepath, arcname)

# Criar o ZIP
print('Criando arquivo compactado com os resultados...')
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # 1. Adicionar o diretório de runs da YOLO
    if os.path.exists(runs_dir):
        zipdir(runs_dir, zipf)
        print('✅ Resultados da YOLO adicionados ao ZIP.')
    else:
        print('⚠️ Pasta de runs da YOLO não encontrada!')
        
    # 2. Adicionar o config.py
    if os.path.exists(config_file):
        zipf.write(config_file, 'config.py')
        print('✅ config.py adicionado ao ZIP.')
        
    # 3. Adicionar o dataset.yaml
    if os.path.exists(dataset_yaml):
        zipf.write(dataset_yaml, 'dataset.yaml')
        print('✅ dataset.yaml adicionado ao ZIP.')

print(f'\nZip criado com sucesso em: {zip_name}')
print('Clique no link abaixo para fazer o download dos resultados:')
display(FileLink(zip_name))